# 20 · Embedding 与距离度量

> **学习目标**：搞清「归一化 / 不归一化」「cosine vs L2 vs dot」三种距离的关系，量化「维度对召回 / 存储 / 检索时间」的影响。
>
> **预备**：notebook 04（向量数学）、09（mini-vecdb）、19（chunking）。
>
> **为什么重要**：RAG 工程师每天和向量打交道。被「为什么 chroma 用 cosine 我用 L2 结果不一样」「为什么 query 越来越慢」绊住时，差别就来自本 notebook 这 4 个点。

In [ ]:
import numpy as np, time, requests
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
np.random.seed(0)
OLLAMA = 'http://127.0.0.1:11434'

## 1. 归一化 = 把向量「投到单位球面」

**定义**：`x / ||x||`，结果长度恒为 1，只保留方向。

**为什么 RAG 默认归一化**：归一化后**点积 = 余弦相似度**，省一次除法 × N。也避免「某些向量天然范数大」干扰排序。

In [ ]:
# 制造一批范数差异很大的向量（模拟「同义但不同写法的句子」embedding 的真实情况）
raw = np.random.randn(5, 16) * np.array([[1, 2, 3, 5, 10]]).T   # 5 个向量，范数依次递增
norms = np.linalg.norm(raw, axis=1)
print(f'原始向量范数: {norms.round(2)}')

# 同一 query
q = np.random.randn(16)

# 不归一化 → 点积
scores_dot = raw @ q
print(f'\n不归一化 点积排序: {(-scores_dot).argsort().tolist()}')
print(f'  → 「范数大的向量天生分高」，排序被范数主导而非真实方向相似度')

# 归一化后
normalized = raw / norms[:, None]
q_norm = q / np.linalg.norm(q)
scores_cos = normalized @ q_norm
print(f'\n归一化 余弦排序  : {(-scores_cos).argsort().tolist()}')
print(f'  → 只看方向相似度，公平')

## 2. cosine / L2 / dot 三种距离 —— 归一化后两两等价

**关键公式**（向量都已归一化时）：
$$\|a - b\|_2^2 = \|a\|^2 + \|b\|^2 - 2 a \cdot b = 1 + 1 - 2 \cos(a, b) = 2 - 2\cos(a, b)$$

**所以**：归一化后，**按 L2 距离排序 ≡ 按 cosine 倒序排序 ≡ 按点积倒序排序**。

In [ ]:
n = 20
d = 64
X = np.random.randn(n, d).astype(np.float32)
X = X / np.linalg.norm(X, axis=1, keepdims=True)
q = np.random.randn(d).astype(np.float32)
q = q / np.linalg.norm(q)

cos  = X @ q                                  # 范围 [-1, 1]，大 = 近
l2sq = np.sum((X - q[None, :]) ** 2, axis=1)   # L2 距离平方，小 = 近
dot  = X @ q                                   # 同 cos（已归一化）

# 三种排序（按「相似度从高到低」）
order_cos  = np.argsort(-cos)
order_l2   = np.argsort(l2sq)
order_dot  = np.argsort(-dot)

print('top-10 排序对比（归一化前提下应完全一致）:')
print('  cos:', order_cos[:10])
print('  L2 :', order_l2[:10])
print('  dot:', order_dot[:10])

assert np.array_equal(order_cos, order_l2),  'cos 与 L2 排序应等价'
assert np.array_equal(order_cos, order_dot), 'cos 与 dot 排序应等价'
print('\n✅ 三种距离排序完全一致 —— 归一化后选哪个都行')

# 数学验证：l2sq ≈ 2 - 2*cos
rhs = 2 - 2 * cos
diff = np.abs(l2sq - rhs).max()
print(f'\n|L2² - (2 - 2cos)|max = {diff:.2e}（应近 0）')

In [ ]:
# 不归一化时 ── 三种距离会得到不同排序
X_unnorm = np.random.randn(n, d).astype(np.float32) * np.linspace(1, 10, n)[:, None]

cos_u  = (X_unnorm @ q) / np.linalg.norm(X_unnorm, axis=1)
l2sq_u = np.sum((X_unnorm - q[None, :]) ** 2, axis=1)
dot_u  = X_unnorm @ q

order_cos_u = np.argsort(-cos_u)[:5]
order_l2_u  = np.argsort(l2sq_u)[:5]
order_dot_u = np.argsort(-dot_u)[:5]

print('不归一化下 top-5（应不一样）:')
print('  cos:', order_cos_u)
print('  L2 :', order_l2_u)
print('  dot:', order_dot_u)
print('\n→ 实战建议：要么写入时归一化（推荐），要么数据库选 cosine。**不要随便切距离函数**。')

## 3. 维度 d 的三角权衡：召回 vs 存储 vs 检索时间

**经验关系**：
- 维度 ↑ → embedding 表达力 ↑ → 召回 ↑（边际递减）
- 维度 ↑ → 存储 = N × d × 4B（fp32）线性涨
- 维度 ↑ → brute force 检索 O(N·d) 线性慢，HNSW 影响轻微但建索引慢

In [ ]:
# 实测：100k 向量 brute force 检索 5 个 query，维度从 64 到 1024 的时间
N = 100_000
dims = [64, 128, 256, 512, 768, 1024]
n_query = 5

results = []
for d in dims:
    X = np.random.randn(N, d).astype(np.float32)
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    Q = np.random.randn(n_query, d).astype(np.float32)
    Q = Q / np.linalg.norm(Q, axis=1, keepdims=True)
    storage_mb = X.nbytes / 1024 / 1024

    t0 = time.perf_counter()
    for _ in range(3):       # 跑 3 次取平均
        _ = Q @ X.T
    dt = (time.perf_counter() - t0) / 3 * 1000
    results.append({'d': d, 'storage_mb': storage_mb, 'query_ms': dt})
    print(f'd={d:>4}  存储={storage_mb:>6.1f} MB   {n_query} query brute = {dt:.1f} ms')

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ds = [r['d'] for r in results]
ax1.plot(ds, [r['storage_mb'] for r in results], '-o', color='C0', label='存储 (MB)')
ax1.set_xlabel('embedding dim'); ax1.set_ylabel('存储 (MB)', color='C0')
ax2 = ax1.twinx()
ax2.plot(ds, [r['query_ms'] for r in results], '-s', color='C1', label='5-query 时间 (ms)')
ax2.set_ylabel('5-query brute force 时间 (ms)', color='C1')
plt.title(f'维度对存储 / 检索时间的影响 (N={N:,})')
plt.grid(True); plt.show()

print('\n经验法则：')
print('  768 维 = 当前 RAG 主流的「甜点」（nomic-embed-text / bge-small）')
print('  1024-1536 维 = OpenAI text-embedding-3-large / bge-large')
print('  3072 维 = OpenAI text-embedding-3-large 满配，存储 / 检索成本明显高')
print('  超 4096 维 = 多模态 / 多语言旗舰，**通常配合 Matryoshka 压维** 用')

## 4. 真实 embedding：本机 Ollama nomic-embed-text

**先 `ollama serve` 再跑下面 cell**。看真实 embedding 对「同义改写」的稳定性 —— 这是 fake_embed 永远做不到的。

In [ ]:
def ollama_up() -> bool:
    try: requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status(); return True
    except Exception: return False

def ollama_embed(text: str, model: str = 'nomic-embed-text') -> np.ndarray:
    r = requests.post(f'{OLLAMA}/api/embeddings', json={'model': model, 'prompt': text}, timeout=30)
    r.raise_for_status()
    v = np.array(r.json()['embedding'], dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

if not ollama_up():
    print('⚠ Ollama 未启动 —— 跳过本节。请在另一终端 `ollama serve` 后重跑。')
else:
    pairs = [
        ('Transformer 是什么',          '注意力机制的基础架构'),
        ('RAG 解决什么问题',            '检索增强生成解决幻觉'),
        ('Transformer 是什么',          '今天天气真好'),    # 不相关
        ('LoRA 微调',                    'QLoRA 量化微调'),
    ]
    print(f'{"句 A":<25} {"句 B":<25} {"cos":>8}')
    print('-' * 65)
    for a, b in pairs:
        sim = float(ollama_embed(a) @ ollama_embed(b))
        print(f'{a:<25} {b:<25} {sim:>+8.4f}')
    print('\n→ 同义改写 cos 应 > 0.5；不相关 cos 应 < 0.3。真 embedding 看到的就是「方向」。')

## 5. 中文 vs 英文 embedding 选型速查

| 场景 | 推荐 | 维度 | 备注 |
|------|------|------|------|
| 英文短文本通用 | `text-embedding-3-small` (OpenAI) / `nomic-embed-text` | 768–1536 | 闭源贵但准；nomic 完全本地 |
| 中文文本 | `bge-m3` / `bge-small-zh-v1.5` / `Conan-embedding` | 768–1024 | bge-m3 多语言 + 多功能（dense + sparse + multi-vec） |
| 多语言混合 | `bge-m3` / `text-embedding-3-large` | 1024–3072 | 强推 bge-m3 |
| 长文档（>2k tokens） | `jina-embeddings-v2-base-en` / `bge-m3` | 768 | 一般 embedding 模型只支持 512 token，长文要专门模型 |
| 多模态（图 + 文） | `jina-clip-v1` / `nomic-embed-vision` | 768 | CLIP-style，同空间下文图可比 |

**本机 Ollama 可用**：`nomic-embed-text`（已装）+ `bge-m3`（需要 `ollama pull bge-m3` 联网）。

## 深入思考

1. **如果两个 chunk embedding cos = 0.99，意味着它们重复吗？**
   - 几乎重复（同一段话不同小改写 / 拷贝粘贴）。生产 RAG 应**入库前去重**：cos > 0.95 视作重复，留一份。
2. **embedding 是不是越新越好？**
   - 不是。新模型可能维度大、推理慢、对 niche 领域不一定准。**先在自己的 eval set 上测**再决定。
3. **重新换 embedding 后，向量库需要重建吗？**
   - 必须。**不同模型的向量空间不可比**（即使维度相同）。换 embedding = 全量重 build。这就是为什么 embedding 选型要慎重。
4. **GPU 上做 embedding 比 CPU 快多少？**
   - 数十倍。但小批量时 CPU 已经够。**embedding 主要瓶颈在 ingest 阶段，不在 query**（query 只 embed 1 个）。
5. **Matryoshka embedding 是什么？**
   - 「俄罗斯套娃」式训练：训练时让模型同时优化「前 K 维」的子向量也有意义。结果：你可以**只用前 512/256 维**做粗检，再用全 1536 维精排，权衡速度 / 精度。

**改一改**：在 ONLINE 模式跑 cell `s4-c1`，把 4 对句子改成你自己工作中真实会问的问题，看 cos 值是否符合直觉。

## 自检 ✅

- [ ] 默写「归一化后 cos / L2 / dot 三者排序等价」的数学证明。
- [ ] 解释「为什么 RAG 默认写入时归一化」。
- [ ] 给一个 100 万向量 × 1024 维的库，能算出存储约 4 GB。
- [ ] 给「换 embedding 模型后召回变烂」的报告，能立刻问「重 build 了吗？eval set 比对过吗？」
- [ ] 解释「Matryoshka embedding 让你能做什么」。

## 下一步

→ [`21_hybrid_search_handrolled.ipynb`](21_hybrid_search_handrolled.ipynb)